# Interactive Credit Analytics Dashboard
An end-to-end interactive dashboard demonstrating data cleaning, preprocessing, normalization, and visualization on credit application data.

In [20]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import ipywidgets as widgets
from IPython.display import display

sns.set_theme(style="whitegrid")
np.random.seed(42)
n_samples = 150

data = {
    'Gender': np.random.choice(['Male', 'Female', None], size=n_samples, p=[0.7, 0.25, 0.05]),
    'Married': np.random.choice(['Yes', 'No', None], size=n_samples, p=[0.6, 0.35, 0.05]),
    'Dependents': np.random.choice(['0', '1', '2', '3+', None], size=n_samples, p=[0.5, 0.2, 0.15, 0.1, 0.05]),
    'Education': np.random.choice(['Graduate', 'Not Graduate', None], size=n_samples, p=[0.75, 0.2, 0.05]),
    'Self_Employed': np.random.choice(['Yes', 'No', None], size=n_samples, p=[0.12, 0.8, 0.08]),
    'ApplicantIncome': np.random.randint(1500, 10000, size=n_samples),
    'CoapplicantIncome': np.random.randint(0, 5000, size=n_samples),
    'LoanAmount': np.random.choice([np.nan, 50, 100, 120, 150, 200, 250, 300], size=n_samples, p=[0.1, 0.1, 0.15, 0.15, 0.2, 0.15, 0.1, 0.05]),
    'Loan_Amount_Term': np.random.choice([360, 180, 240, 120, None], size=n_samples, p=[0.8, 0.08, 0.05, 0.02, 0.05]),
    'Credit_History': np.random.choice([1.0, 0.0, None], size=n_samples, p=[0.8, 0.15, 0.05]),
    'Property_Area': np.random.choice(['Urban', 'Semiurban', 'Rural'], size=n_samples)
}

df = pd.DataFrame(data)
df = pd.concat([df, df.iloc[:5]], ignore_index=True)

for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    df[col] = df[col].fillna(df[col].mode()[0])

df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])

pd.set_option('future.no_silent_downcasting', True)
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

df.drop_duplicates(inplace=True)
df['Dependents'] = df['Dependents'].replace('3+', 3).fillna(0).astype(int)

for col in ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']:
    df[col] = df[col].astype(str).str.strip().str.capitalize()

min_max_scaler = MinMaxScaler()
scale_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount']
df_scaled = df.copy()
df_scaled[scale_cols] = min_max_scaler.fit_transform(df[scale_cols])

df['Loan_to_Income_Ratio'] = df['LoanAmount'] / (df['ApplicantIncome'] / 1000.0)

income_slider = widgets.IntRangeSlider(
    value=[2000, 8000],
    min=1500,
    max=10000,
    step=500,
    description='Income Range:',
    style={'description_width': 'initial'}
)

property_dropdown = widgets.Dropdown(
    options=['All', 'Urban', 'Semiurban', 'Rural'],
    value='All',
    description='Property Location:',
    style={'description_width': 'initial'}
)

ui = widgets.HBox([income_slider, property_dropdown])

def update_dashboard(income_range, location):
    filtered_df = df[(df['ApplicantIncome'] >= income_range[0]) & (df['ApplicantIncome'] <= income_range[1])]
    if location != 'All':
        filtered_df = filtered_df[filtered_df['Property_Area'] == location]

    print(f"--- Showing {len(filtered_df)} Matching Credit Records ---")

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle('Interactive Credit Analytics Dashboard', fontsize=22, fontweight='bold', color='#1a365d', y=0.98)

    sns.regplot(data=filtered_df, x='ApplicantIncome', y='LoanAmount', ax=axes[0, 0], color='#2b6cb0', scatter_kws={'s': 65, 'alpha': 0.75, 'color': '#2b6cb0'}, line_kws={'color': '#e53e3e', 'linewidth': 2})
    axes[0, 0].set_title("Applicant Income vs. Requested Loan Amount", fontsize=13, fontweight='bold', pad=10)
    axes[0, 0].set_xlabel("Applicant Income ($)", fontsize=11)
    axes[0, 0].set_ylabel("Loan Amount ($k)", fontsize=11)

    sns.barplot(data=filtered_df, x='Education', y='ApplicantIncome', ax=axes[0, 1], palette='Blues_r', errorbar=None, edgecolor='#2d3748', linewidth=1)
    axes[0, 1].set_title("Average Income by Education Level", fontsize=13, fontweight='bold', pad=10)
    axes[0, 1].set_xlabel("Education Level", fontsize=11)
    axes[0, 1].set_ylabel("Average Income ($)", fontsize=11)

    credit_counts = filtered_df['Credit_History'].value_counts().rename({1.0: 'Good History', 0.0: 'Bad History'})
    if not credit_counts.empty:
        axes[1, 0].pie(credit_counts, labels=credit_counts.index, autopct='%1.1f%%', colors=['#48bb78', '#f56565'], startangle=140, explode=[0.05]*len(credit_counts), textprops={'fontweight': 'bold', 'fontsize': 11})
    axes[1, 0].set_title("Credit History Status Distribution", fontsize=13, fontweight='bold', pad=10)

    if len(filtered_df) > 1:
        sns.violinplot(data=filtered_df, x='Property_Area', y='Loan_to_Income_Ratio', ax=axes[1, 1], palette='Set2', inner='quartile', linewidth=1.5)
    else:
        sns.scatterplot(data=filtered_df, x='Property_Area', y='Loan_to_Income_Ratio', ax=axes[1, 1], s=120, color='#ed8936', edgecolor='black', zorder=5)
    axes[1, 1].set_title("Loan-to-Income Ratio by Property Area", fontsize=13, fontweight='bold', pad=10)
    axes[1, 1].set_xlabel("Property Area", fontsize=11)
    axes[1, 1].set_ylabel("Loan / (Income / 1000)", fontsize=11)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

out = widgets.interactive_output(update_dashboard, {'income_range': income_slider, 'location': property_dropdown})
display(ui, out)

Output()